# 第13章：回测报告与分析

## 本章学习目标

- 生成专业回测报告
- 分析绩效指标
- 进行收益归因
- 掌握风险指标计算

---

## 13.1 回测报告概述

Qlib 提供了完整的回测报告功能，帮助深入分析策略表现。

### 报告内容

```
回测报告
├── 收益分析
│   ├── 累计收益曲线
│   ├── 年化收益
│   └── 超额收益
├── 风险分析
│   ├── 波动率
│   ├── 最大回撤
│   └── VaR
├── 风险调整收益
│   ├── 夏普比率
│   ├── 信息比率
│   └── Calmar比率
└── 归因分析
    ├── 行业归因
    └── 因子归因
```

In [ ]:
import qlib
from qlib.contrib.report import analysis_position, report_graph
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 初始化 qlib
qlib.init(
    provider_uri="~/.qlib/qlib_data/cn_data",
    region="cn",
)

print("Qlib 初始化成功")

## 13.2 准备回测数据

In [ ]:
# 生成模拟回测数据
np.random.seed(42)

# 时间范围
dates = pd.date_range("2020-01-01", "2022-12-31", freq="B")
n_days = len(dates)

# 模拟策略收益
strategy_returns = np.random.randn(n_days) * 0.015 + 0.0008
benchmark_returns = np.random.randn(n_days) * 0.012 + 0.0004

# 创建 DataFrame
report_df = pd.DataFrame({
    'return': strategy_returns,
    'bench': benchmark_returns,
}, index=dates)

# 计算累计收益
report_df['cum_return'] = (1 + report_df['return']).cumprod()
report_df['cum_bench'] = (1 + report_df['bench']).cumprod()

print(f"数据范围: {dates[0].date()} ~ {dates[-1].date()}")
print(f"交易日数: {n_days}")

## 13.3 绩效指标计算

In [ ]:
def calculate_performance_metrics(returns, benchmark_returns=None, risk_free_rate=0.03):
    """
    计算完整的绩效指标
    
    参数:
        returns: 策略日收益率
        benchmark_returns: 基准日收益率
        risk_free_rate: 无风险利率（年化）
    """
    metrics = {}
    
    # 基本信息
    metrics['交易天数'] = len(returns)
    
    # 收益指标
    total_return = (1 + returns).prod() - 1
    annual_return = (1 + returns.mean()) ** 252 - 1
    metrics['总收益'] = total_return
    metrics['年化收益'] = annual_return
    
    # 风险指标
    annual_vol = returns.std() * np.sqrt(252)
    metrics['年化波动率'] = annual_vol
    
    # 下行风险
    downside_returns = returns[returns < 0]
    downside_std = downside_returns.std() * np.sqrt(252)
    metrics['下行风险'] = downside_std
    
    # 最大回撤
    cum = (1 + returns).cumprod()
    running_max = cum.cummax()
    drawdown = (cum - running_max) / running_max
    max_drawdown = drawdown.min()
    metrics['最大回撤'] = max_drawdown
    
    # 风险调整收益
    excess_return = annual_return - risk_free_rate
    metrics['夏普比率'] = excess_return / annual_vol
    metrics['Sortino比率'] = excess_return / downside_std
    metrics['Calmar比率'] = annual_return / abs(max_drawdown)
    
    # VaR 和 CVaR
    var_95 = np.percentile(returns, 5)
    cvar_95 = returns[returns <= var_95].mean()
    metrics['VaR(95%)'] = var_95
    metrics['CVaR(95%)'] = cvar_95
    
    # 基准相关指标
    if benchmark_returns is not None:
        excess_daily = returns - benchmark_returns
        tracking_error = excess_daily.std() * np.sqrt(252)
        information_ratio = (excess_daily.mean() * 252) / tracking_error
        
        # Beta
        covariance = np.cov(returns, benchmark_returns)[0, 1]
        variance = np.var(benchmark_returns)
        beta = covariance / variance
        
        # Alpha
        alpha = annual_return - risk_free_rate - beta * (benchmark_returns.mean() * 252 - risk_free_rate)
        
        metrics['跟踪误差'] = tracking_error
        metrics['信息比率'] = information_ratio
        metrics['Beta'] = beta
        metrics['Alpha'] = alpha
        metrics['年化超额收益'] = excess_daily.mean() * 252
    
    return metrics

# 计算指标
metrics = calculate_performance_metrics(
    report_df['return'], 
    report_df['bench']
)

print("绩效指标:")
print("=" * 50)
for k, v in metrics.items():
    if isinstance(v, float):
        if '比率' in k or 'Beta' in k or 'Alpha' in k:
            print(f"{k:15s}: {v:.4f}")
        elif '率' in k or 'VaR' in k or 'CVaR' in k:
            print(f"{k:15s}: {v:.2%}")
        else:
            print(f"{k:15s}: {v:.4f}")
    else:
        print(f"{k:15s}: {v}")

## 13.4 收益曲线分析

In [ ]:
# 绘制收益曲线
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 1. 累计收益
ax1 = axes[0]
ax1.plot(report_df.index, report_df['cum_return'], label='策略', linewidth=1.5)
ax1.plot(report_df.index, report_df['cum_bench'], label='基准', linewidth=1.5, linestyle='--')
ax1.set_title('累计收益曲线')
ax1.set_xlabel('日期')
ax1.set_ylabel('累计收益')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 超额收益
ax2 = axes[1]
excess_cum = report_df['cum_return'] / report_df['cum_bench']
ax2.plot(report_df.index, excess_cum, label='超额收益', linewidth=1.5, color='green')
ax2.axhline(y=1, color='red', linestyle='--', linewidth=1)
ax2.set_title('累计超额收益')
ax2.set_xlabel('日期')
ax2.set_ylabel('相对基准')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. 回撤曲线
ax3 = axes[2]
cum = report_df['cum_return']
running_max = cum.cummax()
drawdown = (cum - running_max) / running_max
ax3.fill_between(report_df.index, drawdown, 0, color='red', alpha=0.3, label='回撤')
ax3.set_title('策略回撤')
ax3.set_xlabel('日期')
ax3.set_ylabel('回撤')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 13.5 风险分析

In [ ]:
# 收益分布分析
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 日收益率分布
ax1 = axes[0, 0]
ax1.hist(report_df['return'], bins=50, edgecolor='black', alpha=0.7, density=True)
ax1.axvline(x=0, color='red', linestyle='--', linewidth=1)
ax1.axvline(x=report_df['return'].mean(), color='green', linestyle='-', linewidth=1, label=f'均值: {report_df["return"].mean():.4f}')
ax1.set_title('日收益率分布')
ax1.set_xlabel('日收益率')
ax1.set_ylabel('密度')
ax1.legend()

# 2. Q-Q 图
ax2 = axes[0, 1]
from scipy import stats
stats.probplot(report_df['return'], dist="norm", plot=ax2)
ax2.set_title('Q-Q 图')

# 3. 滚动波动率
ax3 = axes[1, 0]
rolling_vol = report_df['return'].rolling(20).std() * np.sqrt(252)
ax3.plot(rolling_vol, linewidth=1)
ax3.set_title('20日滚动年化波动率')
ax3.set_xlabel('日期')
ax3.set_ylabel('波动率')
ax3.grid(True, alpha=0.3)

# 4. 滚动夏普比率
ax4 = axes[1, 1]
rolling_return = report_df['return'].rolling(60).mean() * 252
rolling_vol_60 = report_df['return'].rolling(60).std() * np.sqrt(252)
rolling_sharpe = (rolling_return - 0.03) / rolling_vol_60
ax4.plot(rolling_sharpe, linewidth=1)
ax4.axhline(y=0, color='red', linestyle='--', linewidth=1)
ax4.set_title('60日滚动夏普比率')
ax4.set_xlabel('日期')
ax4.set_ylabel('夏普比率')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 月度收益分析
monthly_returns = report_df['return'].resample('ME').apply(lambda x: (1 + x).prod() - 1)

# 创建月度收益热力图
monthly_df = pd.DataFrame({
    'year': monthly_returns.index.year,
    'month': monthly_returns.index.month,
    'return': monthly_returns.values
})

monthly_pivot = monthly_df.pivot(index='year', columns='month', values='return')

import seaborn as sns

plt.figure(figsize=(14, 6))
sns.heatmap(monthly_pivot * 100, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            xticklabels=['1月', '2月', '3月', '4月', '5月', '6月', 
                        '7月', '8月', '9月', '10月', '11月', '12月'])
plt.title('月度收益热力图 (%)')
plt.xlabel('月份')
plt.ylabel('年份')
plt.tight_layout()
plt.show()

## 13.6 持仓分析

In [ ]:
# 模拟持仓数据
np.random.seed(42)

# 模拟持仓股票
stocks = [f'SH600{i:03d}' for i in range(1, 51)]

# 模拟持仓权重
n_periods = 10
holdings = {}

for i in range(n_periods):
    weights = np.random.dirichlet(np.ones(30))  # 随机权重
    selected = np.random.choice(stocks, 30, replace=False)
    holdings[f'period_{i}'] = dict(zip(selected, weights))

# 转换为 DataFrame
holdings_df = pd.DataFrame(holdings).fillna(0)

print("持仓数据示例 (前10只股票):")
holdings_df.head(10)

In [ ]:
# 持仓集中度分析
def analyze_concentration(holdings_df):
    """分析持仓集中度"""
    results = []
    
    for col in holdings_df.columns:
        weights = holdings_df[col][holdings_df[col] > 0]
        
        # 计算集中度指标
        results.append({
            'period': col,
            'num_holdings': len(weights),
            'top1_weight': weights.max(),
            'top5_weight': weights.nlargest(5).sum(),
            'top10_weight': weights.nlargest(10).sum(),
            'herfindahl': (weights ** 2).sum(),  # 赫芬达尔指数
        })
    
    return pd.DataFrame(results)

concentration = analyze_concentration(holdings_df)

print("持仓集中度分析:")
concentration

## 13.7 使用 qlib 报告模块

In [ ]:
# 使用 qlib 内置报告功能
from qlib.contrib.report import analysis_position

print("qlib 报告模块功能:")
print("=" * 50)

# 查看可用的分析函数
import inspect

report_funcs = [name for name, _ in inspect.getmembers(analysis_position, inspect.isfunction)]
for func in report_funcs:
    print(f"  - {func}")

## 13.8 生成报告图表

In [ ]:
# 模拟预测分数
np.random.seed(42)
dates = pd.date_range("2020-01-01", "2022-12-31", freq="B")
stocks = [f'SH600{i:03d}' for i in range(1, 31)]

# 创建预测分数
predictions_list = []
for date in dates[::5]:  # 每5天一个预测
    for stock in stocks:
        predictions_list.append({
            'datetime': date,
            'instrument': stock,
            'score': np.random.randn(),
        })

predictions_df = pd.DataFrame(predictions_list)
predictions_df = predictions_df.set_index(['datetime', 'instrument'])

print(f"预测数据形状: {predictions_df.shape}")

In [ ]:
# IC 分析图
# 计算每日 IC
def calculate_ic_series(predictions, labels):
    """计算 IC 序列"""
    ic_list = []
    
    dates = predictions.index.get_level_values('datetime').unique()
    
    for date in dates:
        pred_day = predictions.xs(date, level='datetime')
        label_day = labels.xs(date, level='datetime') if date in labels.index.get_level_values('datetime') else None
        
        if label_day is not None and len(pred_day) > 10:
            common_idx = pred_day.index.intersection(label_day.index)
            if len(common_idx) > 10:
                ic = pred_day.loc[common_idx]['score'].corr(label_day.loc[common_idx])
                ic_list.append({'date': date, 'ic': ic})
    
    return pd.DataFrame(ic_list).set_index('date')

# 模拟标签
labels_list = []
for date in dates[::5]:
    for stock in stocks:
        labels_list.append({
            'datetime': date,
            'instrument': stock,
            'label': np.random.randn(),
        })

labels_df = pd.DataFrame(labels_list)
labels_df = labels_df.set_index(['datetime', 'instrument'])

# 计算 IC
ic_series = calculate_ic_series(predictions_df, labels_df)

# 绘制 IC 图
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# IC 时序
axes[0].bar(ic_series.index, ic_series['ic'], 
            color=['green' if x > 0 else 'red' for x in ic_series['ic']],
            alpha=0.7)
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].axhline(y=ic_series['ic'].mean(), color='blue', linestyle='--', 
                label=f'平均 IC: {ic_series["ic"].mean():.4f}')
axes[0].set_title('IC 时序图')
axes[0].set_xlabel('日期')
axes[0].set_ylabel('IC')
axes[0].legend()

# IC 累计
axes[1].plot(ic_series.index, ic_series['ic'].cumsum(), linewidth=1.5)
axes[1].set_title('累计 IC')
axes[1].set_xlabel('日期')
axes[1].set_ylabel('累计 IC')

plt.tight_layout()
plt.show()

print(f"\nIC 统计:")
print(f"  平均 IC: {ic_series['ic'].mean():.4f}")
print(f"  IC 标准差: {ic_series['ic'].std():.4f}")
print(f"  ICIR: {ic_series['ic'].mean() / ic_series['ic'].std():.4f}")

## 13.9 实践练习

In [ ]:
# 练习1: 计算完整的绩效报告
# 包括所有收益指标、风险指标、风险调整收益指标

# 你的代码



# 提示：使用 calculate_performance_metrics 函数

In [ ]:
# 练习2: 实现收益归因分析
# 分析收益来源：选股贡献 vs 择时贡献

# 你的代码



# 提示：
# 选股贡献 = sum(权重 * (个股收益 - 基准收益))
# 择时贡献 = sum((权重 - 基准权重) * 基准收益)

In [ ]:
# 练习3: 分析不同市场环境下的表现
# 将样本分为牛市、熊市、震荡市，分别计算指标

# 你的代码



# 提示：根据基准收益划分市场环境

## 13.10 本章小结

本章我们学习了：

1. **绩效指标计算**：
   - 收益指标：年化收益、超额收益
   - 风险指标：波动率、最大回撤、VaR
   - 风险调整收益：夏普比率、信息比率

2. **收益曲线分析**：
   - 累计收益
   - 超额收益
   - 回撤曲线

3. **风险分析**：
   - 收益分布
   - 滚动波动率
   - 月度收益

4. **持仓分析**：
   - 集中度分析
   - 换手率

### 关键指标速查

| 指标 | 公式 | 说明 |
|------|------|------|
| 夏普比率 | (Rp - Rf) / σp | 单位风险超额收益 |
| 信息比率 | (Rp - Rb) / TE | 单位跟踪误差超额收益 |
| 最大回撤 | max(Peak - Valley) / Peak | 历史最大亏损 |
| Calmar比率 | Rp / MaxDD | 收益回撤比 |

### 下一章预告

下一章我们将学习强化学习交易，包括：
- 强化学习框架
- 环境与模拟器设计
- 训练 RL 策略